# Create RZS NSW Research Grant Awards

Creates OpenAlex award rows from official Royal Zoological Society of New South Wales (RZS NSW) Paddy Pallin Science Grant and Ethel Mary Read Research Grant awardee pages.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/rzsnsw_research_grants_to_s3.py` to download the official RZS NSW pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:** official RZS NSW program and awardee pages under `https://www.rzsnsw.org.au/`  
**S3 location:** `s3a://openalex-ingest/awards/rzsnsw/rzsnsw_research_grants.parquet`  
**Local full-source validation on 2026-05-28:** 82 rows, 2013-2025; 47 Paddy Pallin Science Grant rows and 35 Ethel Mary Read Research Grant rows. Amount/currency are NULL by source authority because the official program pages publish maximum grants/caps, not exact per-recipient amounts.

**Royal Zoological Society of New South Wales funder:**
- funder_id: 4320331891
- OpenAlex ID: `https://openalex.org/F4320331891`
- display_name: `Royal Zoological Society of New South Wales`
- country_code: `AU`
- ror_id: NULL
- doi: `10.13039/501100022897`
- provenance: `rzsnsw_research_grants`
- priority: 182

**Amount/currency decision:** waived under section 6.7. RZS NSW publishes program caps (`up to AUD 10,000` and `maximum AUD 3,000`) but not exact recipient amounts, so the ingest preserves `program_amount_text` and keeps `amount`/`currency` NULL rather than converting caps into award amounts. This follows the same source-authority pattern used for AOS LACCR and NSF Sri Lanka GMIS rows.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rzsnsw_research_grants_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rzsnsw/rzsnsw_research_grants.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 82 rows.
SELECT COUNT(*) as total_rzsnsw_research_grants
FROM openalex.awards.rzsnsw_research_grants_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.rzsnsw_research_grants_raw;


In [ ]:
%sql
-- Sample raw RZS NSW data.
SELECT
    funder_award_id,
    recipient_name,
    affiliation,
    project_title,
    award_year,
    funder_scheme,
    amount,
    currency,
    program_amount_text,
    source_funding_partner,
    landing_page_url
FROM openalex.awards.rzsnsw_research_grants_raw
ORDER BY TRY_CAST(award_year AS INT), funder_scheme, recipient_name
LIMIT 40;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys


In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Uses information_schema, not DESCRIBE as a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'rzsnsw_research_grants_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|award|donation|support|stipend|prize';


In [ ]:
%sql
-- Currency-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'rzsnsw_research_grants_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
-- Confirm coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(funder_award_id) AS rows_with_native_id,
    COUNT(recipient_name) AS rows_with_recipient_name,
    COUNT(given_name) AS rows_with_given_name,
    COUNT(family_name) AS rows_with_family_name,
    COUNT(affiliation) AS rows_with_affiliation,
    COUNT(project_title) AS rows_with_project_title,
    COUNT(amount) AS rows_with_exact_amount,
    COUNT(currency) AS rows_with_currency,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_exact_amount,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year,
    collect_set(currency) AS exact_amount_currencies
FROM openalex.awards.rzsnsw_research_grants_raw;


In [ ]:
%sql
-- Native-key inspection. funder_award_id must be unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_hash) AS distinct_source_hashes,
    COUNT(DISTINCT recipient_name) AS distinct_recipients
FROM openalex.awards.rzsnsw_research_grants_raw;


In [ ]:
%sql
-- Scheme and funding-partner distribution.
SELECT
    funder_scheme,
    source_funding_partner,
    COUNT(*) AS rows,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year
FROM openalex.awards.rzsnsw_research_grants_raw
GROUP BY funder_scheme, source_funding_partner
ORDER BY funder_scheme, rows DESC, source_funding_partner;


In [ ]:
%sql
-- Program cap text preserved by the scraper. These caps are not exact award amounts.
SELECT
    funder_scheme,
    program_amount_text,
    amount_note,
    COUNT(*) AS rows
FROM openalex.awards.rzsnsw_research_grants_raw
GROUP BY funder_scheme, program_amount_text, amount_note
ORDER BY funder_scheme;


In [ ]:
%sql
-- Year distribution from the raw source.
SELECT
    TRY_CAST(award_year AS INT) AS award_year,
    funder_scheme,
    COUNT(*) AS rows
FROM openalex.awards.rzsnsw_research_grants_raw
GROUP BY TRY_CAST(award_year AS INT), funder_scheme
ORDER BY award_year, funder_scheme;


## Step 1.6: Funder Existence Fail-Fast


In [ ]:
%sql
-- Must return TRUE. If this fails, STOP: RZS NSW F4320331891 is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Royal Zoological Society of New South Wales F4320331891'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320331891;


In [ ]:
%sql
-- Inspect the funder row used for the award struct.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320331891;


## Step 2: Transform to OpenAlex Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rzsnsw_research_grants_awards
USING delta
AS
WITH
rzsnsw_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320331891
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        NULLIF(TRIM(currency), '') AS parsed_currency,
        TRY_CAST(award_year AS INT) AS parsed_award_year,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(CONCAT(award_year, '-12-31'), 'yyyy-MM-dd') AS parsed_end_date
    FROM openalex.awards.rzsnsw_research_grants_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
      AND recipient_name IS NOT NULL
      AND TRIM(recipient_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        NULLIF(TRIM(g.funding_type), '') as funding_type,
        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'rzsnsw_research_grants' as provenance,

        CASE
            WHEN g.parsed_award_year > YEAR(current_date()) + 1 THEN NULL
            ELSE g.parsed_start_date
        END as start_date,
        CASE
            WHEN g.parsed_award_year > YEAR(current_date()) + 1 THEN NULL
            ELSE g.parsed_end_date
        END as end_date,
        CASE
            WHEN g.parsed_award_year > YEAR(current_date()) + 1 THEN NULL
            ELSE g.parsed_award_year
        END as start_year,
        CASE
            WHEN g.parsed_award_year > YEAR(current_date()) + 1 THEN NULL
            ELSE g.parsed_award_year
        END as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            CASE
                WHEN g.parsed_award_year > YEAR(current_date()) + 1 THEN NULL
                ELSE g.parsed_start_date
            END as role_start,
            struct(
                NULLIF(TRIM(g.affiliation), '') as name,
                NULLIF(TRIM(g.country), '') as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        NULLIF(TRIM(g.doi), '') as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN rzsnsw_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 182


In [ ]:
%sql
-- Remove previous RZS NSW research grant award data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rzsnsw_research_grants' AND priority = 182;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    182 as priority
FROM openalex.awards.rzsnsw_research_grants_awards;


## Step 6: Verification Queries


In [ ]:
%sql
SELECT COUNT(*) as total_rzsnsw_research_grants
FROM openalex.awards.rzsnsw_research_grants_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.rzsnsw_research_grants_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    lead_investigator.affiliation.name AS source_affiliation,
    landing_page_url
FROM openalex.awards.rzsnsw_research_grants_awards
ORDER BY start_year, funder_scheme, display_name
LIMIT 82;


In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.rzsnsw_research_grants_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only Royal Zoological Society of New South Wales.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.rzsnsw_research_grants_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
-- Data completeness and exact amount coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_exact_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.given_name) as has_given_name,
    COUNT(lead_investigator.family_name) as has_family_name,
    COUNT(lead_investigator.affiliation.name) as has_source_affiliation,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_dates,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_exact_amount
FROM openalex.awards.rzsnsw_research_grants_awards;


In [ ]:
%sql
-- Per-scheme exact amount coverage. Both schemes should remain NULL by source authority.
SELECT
    funder_scheme,
    funding_type,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_exact_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_exact_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.rzsnsw_research_grants_awards
GROUP BY funder_scheme, funding_type
ORDER BY funder_scheme;


In [ ]:
%sql
-- Year distribution.
SELECT start_year, funder_scheme, COUNT(*) AS cnt
FROM openalex.awards.rzsnsw_research_grants_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year, funder_scheme;


In [ ]:
%sql
-- Confirm no future-dated rows slipped past the cap.
SELECT COUNT(*) AS rows_after_future_year_cap
FROM openalex.awards.rzsnsw_research_grants_awards
WHERE start_year > YEAR(current_date()) + 1;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 182.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rzsnsw_research_grants' AND priority = 182
GROUP BY priority, provenance;


## Handoff / Admin Notes

Contractor-complete handoff only. The contractor has no S3 or Databricks access, so an admin must:

1. Run `scripts/local/rzsnsw_research_grants_to_s3.py` to upload `s3://openalex-ingest/awards/rzsnsw/rzsnsw_research_grants.parquet`.
2. Run this notebook on Databricks.
3. Run the Step 6 verification cells and QA the inserted `provenance='rzsnsw_research_grants'`, `priority=182` rows.
4. Mark the tracker row Complete after production API verification.

Do not add job YAML until an admin has run and verified the ingest.
